# Logistic Regression
Checking the weights for Dry- and Wet-proofing.
What's the difference?

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
data_NL = pd.read_csv('../../../data/raw/SCALAR_Coastal_Study_new_respondents_Wave_Five_NL.csv')
data_UK = pd.read_csv('../../../data/raw/SCALAR_Coastal_Study_new_respondents_Wave_Five_UK.csv')

For logistic regression on whether households will take measures, we need the following data:
- threat appraisal:
    * perceived probability (R02_perc_prob)
    * perceived damage / severity (R03_perc_damage) - we don't have this so we estimate it based on the perc probability, worry and flood experience
    * worry (R05_worry)
- coping appraisal
    * perceived cost (R1c_perc_costs)
    * perceived response efficacy (R1b_resp_efficacy)
    * self-efficacy (R1a_self_efficacy)
    * each for the following measures
       * dry-proofing:
          * installing anti-backflow valves on pipes (SM5)
          * installing a pump or similar to drain water (SM6)
          * fixing water barriers (SM7)
       * wet-proofing:
          * strengthening house foundations (SM2)
          * reinforcing walls (SM3)
          * raising electricity meter (SM4)
- other:
    * flood experience
 
And for the y: 
- R2_implementation for relevant measures. Reminder: 1: already implemented, 2: plan to implement in near future (next 1-3 years), rest: implement later or never)


In [3]:
relevant_columns = ['R02_perc_prob', 'R05_worry', # threat appraisal
                    'R1a_self_efficacy_SM2', 'R1a_self_efficacy_SM3', 'R1a_self_efficacy_SM4', 'R1a_self_efficacy_SM5', 'R1a_self_efficacy_SM6', 'R1a_self_efficacy_SM7',                    
                    'R1b_resp_efficacy_SM2', 'R1b_resp_efficacy_SM3', 'R1b_resp_efficacy_SM4', 'R1b_resp_efficacy_SM5', 'R1b_resp_efficacy_SM6', 'R1b_resp_efficacy_SM7', 
                    'R1c_perc_cost_SM2', 'R1c_perc_cost_SM3', 'R1c_perc_cost_SM4', 'R1c_perc_cost_SM5', 'R1c_perc_cost_SM6', 'R1c_perc_cost_SM7', 
                    'Q18_flood_exp',
                    'R2_implementation_SM2', 'R2_implementation_SM3', 'R2_implementation_SM4', 'R2_implementation_SM5', 'R2_implementation_SM6', 'R2_implementation_SM7']

### Preparing country dataframes

#### Netherlands

In [4]:
# add column on perceived damage for NL (see estimate_severity_NL.ipynb for equation generation)
data_NL['R03_perc_damage'] = 0.4302 + (0.2665 * data_NL['R02_perc_prob']) + (0.1581 * data_NL['R05_worry']) + (-0.0871 * data_NL['Q18_flood_exp'])
data_NL.head()

,ID,Q1_home_NL_UK,Q4_home_size_NL,Q5_home_tenure,Q5b_home_sell,Q6_home_costs,Q7_move_in,Q8_move_out,Q12_neighborhood_trust,Q13_neighborhood_community,Q14_neighborhood_pleasure,Q15_neighborhood_favorite,Q16_neighborhood_identity,Q11_search_improve,Q11_search_social,Q11_search_family,Q11_search_area,Q11_search_job,Q11_search_location,Q11_search_hazard,Q11_search_other,Q11_search_dont_know,Q11a_hazard_type1,Q11a_hazard_type3,Q11a_hazard_type4,Q11a_hazard_type5,Q11a_hazard_type10,Q11a_hazard_type6,Q11a_hazard_type9,Q11a_hazard_not_say,R02_perc_prob,R02_perc_prob_other_text,Q18_flood_exp,Q18a_flood_where,Q18b_flood_year,Q18d_flood_cost,R05_worry,R01_resilience_5,R01_resilience_6,Q17_compens_noone,Q17_compens_ins,Q17_compens_owner,Q17_compens_family,Q17_compens_ngo,Q17_compens_other,Q17_compens_dont_know,R1a_self_efficacy_SM1,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM1,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM1,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,R2_implementation_SM1,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,R1a_self_efficacy_NM1,R1a_self_efficacy_NM2,R1a_self_efficacy_NM3,R1a_self_efficacy_NM4,R1a_self_efficacy_NM5,R1a_self_efficacy_NM6,R1a_self_efficacy_NM7,R1a_self_efficacy_NM8,R1a_self_efficacy_NM9,R1a_self_efficacy_NM10,R1a_self_efficacy_NM11,R1b_resp_efficacy_NM1,R1b_resp_efficacy_NM2,R1b_resp_efficacy_NM3,R1b_resp_efficacy_NM4,R1b_resp_efficacy_NM5,R1b_resp_efficacy_NM6,R1b_resp_efficacy_NM7,R1b_resp_efficacy_NM8,R1b_resp_efficacy_NM9,R1b_resp_efficacy_NM10,R1b_resp_efficacy_NM11,R1c_perc_cost_NM1,R1c_perc_cost_NM2,R1c_perc_cost_NM3,R1c_perc_cost_NM4,R1c_perc_cost_NM5,R1c_perc_cost_NM6,R1c_perc_cost_NM7,R1c_perc_cost_NM8,R1c_perc_cost_NM9,R1c_perc_cost_NM10,R1c_perc_cost_NM11,R2_implementation_NM1,R2_implementation_NM2,R2_implementation_NM3,R2_implementation_NM4,R2_implementation_NM5,R2_implementation_NM6,R2_implementation_NM7,R2_implementation_NM8,R2_implementation_NM9,R2_implementation_NM10,R2_implementation_NM11,Q41_ins_UK,Q41a_property_NL,Q41a_possessions_NL,Q41b_ins_intention_NL,Q41c_ins_self_efficacy_NL,Q25_fl_convo_times,Q25a_fl_convo_local,Q26_measures_advice,Q27_fl_convo_people,Q28_convo_resp_efficacy,Q28_convo_costs,Q28_convo_prob,Q28_convo_severity,Q28_convo_worry,Q28_convo_community,Q28_convo_gov,Q29a_adapt_p1,Q29a_adapt_p2,Q29a_adapt_p3,Q29d_distance_p1,Q29d_distance_p2,Q29d_distance_p3,Q29f_com_freq_p1,Q29f_com_freq_p2,Q29f_com_freq_p3,Q29g_flood_com_freq_p1,Q29g_flood_com_freq_p2,Q29g_flood_com_freq_p3,Q29i_worry_p1,Q29i_worry_p2,Q29i_worry_p3,Q29j_valuable_exp_p1,Q29j_valuable_exp_p2,Q29j_valuable_exp_p3,Q30_adapt_against,Q31_give_advice,Q32_influence,Q33_ask_advice_family,Q33_ask_advice_neighbor,Q33_ask_advice_colleague,Q33_ask_advice_online,Q33_ask_advice_expert,Q33_ask_advice_teacher,Q33_ask_advice_ngo,Q33_ask_advice_gov,Q33_ask_advice_other,Q33_ask_advice_dont_know,Q33_ask_advice_never,Q33_ask_advice_other_text,Q34_influential_family,Q34_influential_neighbor,Q34_influential_colleague,Q34_influential_online,Q34_influential_expert,Q34_influential_teacher,Q34_influential_ngo,Q34_influential_gov,Q34_influential_other,Q34_influential_dont_know,Q34_influential_noone,Q34_influential_other_text,Q16_compens_gov,Q37_govern,Q38_market,Q39_technology,Q40_nature,Q41a_media_freq,Q41b_s_media_freq,Q42_local_gov_info,Q42a_local_gov_trust,Q42b_local_gov_act,Q43_national_gov_info,Q43a_national_gov_trust,Q43b_national_gov_act,Q44_trust_pm,Q44_trust_gov_rep,Q44_trust_family,Q44_trust_media,Q44_trust_s_media,Q45_policies_relocate,Q45_policies_research,Q45_policies_subsidies,Q45_policies_information,Q45_policies_large_infra,Q45_policies_zoning,Q45_policie

In [5]:
PMT_data_NL = data_NL[relevant_columns + ['R03_perc_damage']]
PMT_data_NL.head()

,R02_perc_prob,R05_worry,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,R03_perc_damage
0,6,2,1,1,3,3,4,1,2,3,2,3,4,2,4,5,5,4,4,3,0,4,4,4,4,3,4,2.3454
1,4,1,3,5,5,5,3,3,3,3,3,3,3,3,3,3,3,4,3,4,0,4,4,4,4,4,4,1.6543
2,2,1,1,1,5,5,1,1,1,1,1,2,3,2,5,5,4,3,4,3,0,4,4,4,4,4,4,1.1213
3,1,1,1,1,1,1,1,1,3,3,3,3,3,3,5,5,5,5,5,5,0,4,4,4,4,4,4,0.8548
4,2,1,1,1,2,2,2,1,1,1,1,1,2,1,5,5,3,3,4,3,0,4,4,4,4,4,4,1.1213


#### United Kingdom

In [6]:
PMT_data_UK = data_UK[relevant_columns + ['R03_perc_damage_UK1', 'R03_perc_damage_UK2', 'R03_perc_damage_UK3']]
PMT_data_UK.head()

,R02_perc_prob,R05_worry,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,R03_perc_damage_UK1,R03_perc_damage_UK2,R03_perc_damage_UK3
0,6,1,1,1,5,4,2,3,1,1,1,1,1,2,5,5,2,2,3,3,0,4,4,4,4,4,4,,98,
1,2,1,1,5,1,3,3,4,1,1,1,1,3,3,5,5,3,2,3,2,0,4,4,4,4,4,4,,1,
2,1,1,1,5,5,5,5,1,5,5,4,5,4,4,5,5,1,1,1,3,0,4,4,4,4,4,4,,1,
3,1,1,1,1,1,1,1,1,5,5,5,5,5,5,5,5,5,5,5,5,0,4,4,4,4,4,4,,1,
4,2,1,2,2,2,2,2,4,2,2,2,2,2,2,5,5,5,5,5,5,1,4,4,4,4,4,4,,1,


In [7]:
PMT_data_UK = PMT_data_UK.replace(r'^\s*$', np.nan, regex=True)
PMT_data_UK['R03_perc_damage_UK1'] = pd.to_numeric(PMT_data_UK['R03_perc_damage_UK1'])
PMT_data_UK['R03_perc_damage_UK2'] = pd.to_numeric(PMT_data_UK['R03_perc_damage_UK2'])
PMT_data_UK['R03_perc_damage_UK3'] = pd.to_numeric(PMT_data_UK['R03_perc_damage_UK3'])
PMT_data_UK['R03_perc_damage'] = PMT_data_UK.R03_perc_damage_UK1.fillna(0) + PMT_data_UK.R03_perc_damage_UK2.fillna(0)+ PMT_data_UK.R03_perc_damage_UK3.fillna(0)
PMT_data_UK = PMT_data_UK.drop(columns = ['R03_perc_damage_UK1', 'R03_perc_damage_UK2', 'R03_perc_damage_UK3'])
PMT_data_UK.head()

,R02_perc_prob,R05_worry,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,R03_perc_damage
0,6,1,1,1,5,4,2,3,1,1,1,1,1,2,5,5,2,2,3,3,0,4,4,4,4,4,4,98.0
1,2,1,1,5,1,3,3,4,1,1,1,1,3,3,5,5,3,2,3,2,0,4,4,4,4,4,4,1.0
2,1,1,1,5,5,5,5,1,5,5,4,5,4,4,5,5,1,1,1,3,0,4,4,4,4,4,4,1.0
3,1,1,1,1,1,1,1,1,5,5,5,5,5,5,5,5,5,5,5,5,0,4,4,4,4,4,4,1.0
4,2,1,2,2,2,2,2,4,2,2,2,2,2,2,5,5,5,5,5,5,1,4,4,4,4,4,4,1.0


### Combining data and moving on

In [8]:
data_households = pd.concat([PMT_data_UK, PMT_data_NL])
data_households.head()

,R02_perc_prob,R05_worry,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,R03_perc_damage
0,6,1,1,1,5,4,2,3,1,1,1,1,1,2,5,5,2,2,3,3,0,4,4,4,4,4,4,98.0
1,2,1,1,5,1,3,3,4,1,1,1,1,3,3,5,5,3,2,3,2,0,4,4,4,4,4,4,1.0
2,1,1,1,5,5,5,5,1,5,5,4,5,4,4,5,5,1,1,1,3,0,4,4,4,4,4,4,1.0
3,1,1,1,1,1,1,1,1,5,5,5,5,5,5,5,5,5,5,5,5,0,4,4,4,4,4,4,1.0
4,2,1,2,2,2,2,2,4,2,2,2,2,2,2,5,5,5,5,5,5,1,4,4,4,4,4,4,1.0


Need to combine some columns. 
- combining dry- and wet-proofing --> averages

In [9]:
data_households['self_efficacy_DP'] = data_households[['R1a_self_efficacy_SM5', 'R1a_self_efficacy_SM6', 'R1a_self_efficacy_SM7']].mean(axis=1)
data_households['self_efficacy_WP'] = data_households[['R1a_self_efficacy_SM2', 'R1a_self_efficacy_SM3', 'R1a_self_efficacy_SM4']].mean(axis=1)
data_households['resp_efficacy_DP'] = data_households[['R1b_resp_efficacy_SM5', 'R1b_resp_efficacy_SM6', 'R1b_resp_efficacy_SM7']].mean(axis=1)
data_households['resp_efficacy_WP'] = data_households[['R1b_resp_efficacy_SM2', 'R1b_resp_efficacy_SM3', 'R1b_resp_efficacy_SM4']].mean(axis=1)
data_households['perc_cost_DP'] = data_households[['R1c_perc_cost_SM5', 'R1c_perc_cost_SM6', 'R1c_perc_cost_SM7']].mean(axis=1)
data_households['perc_cost_WP'] = data_households[['R1c_perc_cost_SM2', 'R1c_perc_cost_SM3', 'R1c_perc_cost_SM4']].mean(axis=1)

In [10]:
data_households.head()

,R02_perc_prob,R05_worry,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,R03_perc_damage,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP
0,6,1,1,1,5,4,2,3,1,1,1,1,1,2,5,5,2,2,3,3,0,4,4,4,4,4,4,98.0,3.000000,2.333333,1.333333,1.000000,2.666667,4.000000
1,2,1,1,5,1,3,3,4,1,1,1,1,3,3,5,5,3,2,3,2,0,4,4,4,4,4,4,1.0,3.333333,2.333333,2.333333,1.000000,2.333333,4.333333
2,1,1,1,5,5,5,5,1,5,5,4,5,4,4,5,5,1,1,1,3,0,4,4,4,4,4,4,1.0,3.666667,3.666667,4.333333,4.666667,1.666667,3.666667
3,1,1,1,1,1,1,1,1,5,5,5,5,5,5,5,5,5,5,5,5,0,4,4,4,4,4,4,1.0,1.000000,1.000000,5.000000,5.000000,5.000000,5.000000
4,2,1,2,2,2,2,2,4,2,2,2,2,2,2,5,5,5,5,5,5,1,4,4,4,4,4,4,1.0,2.666667,2.000000,2.000000,2.000000,5.000000,5.000000


How to combine the implementation for dry- and wet-proofing. Maybe they plan to take some of the dry-proofing emasures but not all. I don't think it makes sense to take the average.
Idea: always take the lowest number available in the three columns that go into the variable (assumption: if they would take 1, they would also do the other)

In [11]:
data_households['implement_DP'] = data_households[['R2_implementation_SM5', 'R2_implementation_SM6', 'R2_implementation_SM7']].min(axis=1)
data_households['implement_WP'] = data_households[['R2_implementation_SM2', 'R2_implementation_SM3', 'R2_implementation_SM4']].min(axis=1)


Cleaning up: removing rows with don't know answers and removing the columns that were used to combine for wet- and dry-proofing just above. 

In [12]:
# remove rows with don't know
data_households_98_removed = data_households[(data_households.R02_perc_prob != 95) &
                                            (data_households.R02_perc_prob != 98) & 
                                            (data_households.R02_perc_prob != 97) & 
                                            (data_households.R03_perc_damage != 98) &
                                            (data_households.R05_worry != 98)
                                            ]
data_households_98_removed.describe()

,R02_perc_prob,R05_worry,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,Q18_flood_exp,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,R03_perc_damage,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP
count,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000,922.000000
mean,2.398048,1.543384,1.731020,1.813449,2.157267,2.180043,2.100868,1.906725,2.532538,2.552061,2.767896,2.521692,2.716920,2.578091,4.411063,4.331887,3.590022,3.610629,3.845987,3.810195,0.131236,3.821041,3.812364,3.688720,3.732104,3.790672,3.797180,1.757520,2.062545,1.900578,2.605568,2.617498,3.755604,4.110991,3.647505,3.629067
std,1.861933,0.869788,1.140172,1.190769,1.420417,1.335278,1.292095,1.244304,1.213176,1.196267,1.318181,1.133857,1.183086,1.228895,0.938504,0.929924,1.189452,1.104337,1.051409,1.113428,0.337842,0.575911,0.593613,0.790595,0.698089,0.609907,0.622656,1.069016,1.138208,1.095565,1.048701,1.110058,0.966464,0.843301,0.819443,0.866270
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.767700,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000,2.000000,1.000000,4.000000,4.000000,3.000000,3.000000,3.000000,3.000000,0.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,1.000000,1.000000,1.000000,2.000000,1.666667,3.000000,3.666667,4.000000,4.000000
50%,2.000000,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000,1.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,5.000000,5.000000,4.000000,4.000000,4.000000,4.000000,0.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,1.121300,1.666667,1.333333,2.666667,2.666667,3.666667,4.333333,4.000000,4.000000
75%,3.000000,2.000000,2.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,4.000000,3.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,0.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,2.000000,3.000000,2.666667,3.000000,3.333333,4.666667,5.000000,4.000000,4.000000
max,9.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,1.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,4.000000,4.000000


In [13]:
data_households_98_removed = data_households_98_removed.drop(columns = [#'R03_perc_damage_UK1', 'R03_perc_damage_UK2', 'R03_perc_damage_UK3',# threat appraisal
                    'R1a_self_efficacy_SM2', 'R1a_self_efficacy_SM3', 'R1a_self_efficacy_SM4', 'R1a_self_efficacy_SM5', 'R1a_self_efficacy_SM6', 'R1a_self_efficacy_SM7',                    
                    'R1b_resp_efficacy_SM2', 'R1b_resp_efficacy_SM3', 'R1b_resp_efficacy_SM4', 'R1b_resp_efficacy_SM5', 'R1b_resp_efficacy_SM6', 'R1b_resp_efficacy_SM7', 
                    'R1c_perc_cost_SM2', 'R1c_perc_cost_SM3', 'R1c_perc_cost_SM4', 'R1c_perc_cost_SM5', 'R1c_perc_cost_SM6', 'R1c_perc_cost_SM7',
                                          'R2_implementation_SM2', 'R2_implementation_SM3', 'R2_implementation_SM4', 'R2_implementation_SM5', 'R2_implementation_SM6', 'R2_implementation_SM7'
                                                                       ])
data_households_98_removed.head()

,R02_perc_prob,R05_worry,Q18_flood_exp,R03_perc_damage,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP
1,2,1,0,1.0,3.333333,2.333333,2.333333,1.000000,2.333333,4.333333,4,4
2,1,1,0,1.0,3.666667,3.666667,4.333333,4.666667,1.666667,3.666667,4,4
3,1,1,0,1.0,1.000000,1.000000,5.000000,5.000000,5.000000,5.000000,4,4
4,2,1,1,1.0,2.666667,2.000000,2.000000,2.000000,5.000000,5.000000,4,4
5,1,1,0,3.0,1.000000,1.000000,3.000000,3.000000,4.333333,4.666667,4,4


In [14]:
# Initialize MinMaxScaler
scaler = MinMaxScaler()
# Normalize 
data_households_98_removed[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 
                            'self_efficacy_DP', 'self_efficacy_WP', 
                            'resp_efficacy_DP', 'resp_efficacy_WP', 
                            'perc_cost_DP', 'perc_cost_WP']] = scaler.fit_transform(data_households_98_removed[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 
                                                                                                                'self_efficacy_DP', 'self_efficacy_WP', 
                                                                                                                'resp_efficacy_DP', 'resp_efficacy_WP', 
                                                                                                                'perc_cost_DP', 'perc_cost_WP']])
data_households_98_removed.head()

,R02_perc_prob,R05_worry,Q18_flood_exp,R03_perc_damage,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP
1,0.125,0.0,0.0,0.054887,0.583333,0.333333,0.333333,0.000000,0.333333,0.833333,4,4
2,0.000,0.0,0.0,0.054887,0.666667,0.666667,0.833333,0.916667,0.166667,0.666667,4,4
3,0.000,0.0,0.0,0.054887,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,4,4
4,0.125,0.0,1.0,0.054887,0.416667,0.250000,0.250000,0.250000,1.000000,1.000000,4,4
5,0.000,0.0,0.0,0.527444,0.000000,0.000000,0.500000,0.500000,0.833333,0.916667,4,4


In [15]:
# binary columns for implementation
data_households_98_removed['done_DP'] = (data_households_98_removed['implement_DP'] == 1).astype(int)
data_households_98_removed['done_WP'] = (data_households_98_removed['implement_WP'] == 1).astype(int)
data_households_98_removed['plan_soon_DP'] = (data_households_98_removed['implement_DP'] == 2).astype(int)
data_households_98_removed['plan_soon_WP'] = (data_households_98_removed['implement_WP'] == 2).astype(int)
data_households_98_removed.head(10)

,R02_perc_prob,R05_worry,Q18_flood_exp,R03_perc_damage,self_efficacy_DP,self_efficacy_WP,resp_efficacy_DP,resp_efficacy_WP,perc_cost_DP,perc_cost_WP,implement_DP,implement_WP,done_DP,done_WP,plan_soon_DP,plan_soon_WP
1,0.125,0.00,0.0,0.054887,0.583333,0.333333,0.333333,0.000000,0.333333,0.833333,4,4,0,0,0,0
2,0.000,0.00,0.0,0.054887,0.666667,0.666667,0.833333,0.916667,0.166667,0.666667,4,4,0,0,0,0
3,0.000,0.00,0.0,0.054887,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,4,4,0,0,0,0
4,0.125,0.00,1.0,0.054887,0.416667,0.250000,0.250000,0.250000,1.000000,1.000000,4,4,0,0,0,0
5,0.000,0.00,0.0,0.527444,0.000000,0.000000,0.500000,0.500000,0.833333,0.916667,4,4,0,0,0,0
6,0.000,0.00,0.0,0.054887,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,4,4,0,0,0,0
7,0.000,0.00,0.0,0.054887,0.833333,0.000000,0.333333,0.166667,0.500000,0.916667,4,4,0,0,0,0
8,0.000,0.25,0.0,0.054887,0.583333,0.333333,0.583333,0.666667,0.500000,0.750000,4,1,0,1,0,0
9,0.000,0.25,0.0,0.291166,0.750000,0.166667,0.500000,0.000000,0.500000,1.000000,4,4,0,0,0,0
10,0.875,1.00,1.0,0.527444,0.000000,0.000000,0.500000,0.750000,1.000000,1.000000,4,4,0,0,0,0


In [16]:
print(data_households_98_removed['done_DP'].value_counts())
print(data_households_98_removed['done_WP'].value_counts())
print(data_households_98_removed['plan_soon_DP'].value_counts())
print(data_households_98_removed['plan_soon_WP'].value_counts())

done_DP
0    875
1     47
Name: count, dtype: int64
done_WP
0    861
1     61
Name: count, dtype: int64
plan_soon_DP
0    859
1     63
Name: count, dtype: int64
plan_soon_WP
0    867
1     55
Name: count, dtype: int64


In [17]:
data_DP_WP_reg = data_households_98_removed

In [18]:
data_DP_WP_reg.count()

R02_perc_prob       922
R05_worry           922
Q18_flood_exp       922
R03_perc_damage     922
self_efficacy_DP    922
self_efficacy_WP    922
resp_efficacy_DP    922
resp_efficacy_WP    922
perc_cost_DP        922
perc_cost_WP        922
implement_DP        922
implement_WP        922
done_DP             922
done_WP             922
plan_soon_DP        922
plan_soon_WP        922
dtype: int64

### Do the logistic regression

We do the logistic regression 4 times here. 
1. logistic regression to done_WP, so to the ones that have already taken the measure WP
2. logistic regression to plan_soon_WP: those who plan to take the measure in the next 6 months
3. logistic regression to done_DP, so to the ones that have already taken the measure DP
4. logistic regression to plan_soon_DP: those who plan to take the measure in the next 6 months

In [19]:
# Wet-proofing
# logistic regression to 'done_WP'
# Define features (X) and target (y)
X_WP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP']]
y_WP_done = data_DP_WP_reg['done_WP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_WP_done, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_WP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 0.94

Logistic Regression Equation:
logit(p) = -2.3829 (-0.3450 * R02_perc_prob) + (0.8352 * R05_worry) + (0.7249 * Q18_flood_exp) + (0.1587 * R03_perc_damage) + (0.6433 * self_efficacy_WP) + (0.4269 * resp_efficacy_WP) + (-1.4259 * perc_cost_WP) + (1.6349 * done_DP)

Feature Weights: {'R02_perc_prob': -0.3449500676077063, 'R05_worry': 0.8351560365134342, 'Q18_flood_exp': 0.7248682368761468, 'R03_perc_damage': 0.15865663843752514, 'self_efficacy_WP': 0.6432834947530413, 'resp_efficacy_WP': 0.42686885537001773, 'perc_cost_WP': -1.4258860648665674, 'done_DP': 1.634870619083835}
Intercept: -2.382891002312257


In [20]:
LR_values_done_WP = dict(zip(feature_names, weights))
LR_values_done_WP['Intercept'] = intercept
LR_values_done_WP['Perc_probability'] = LR_values_done_WP['R02_perc_prob']
LR_values_done_WP['worry'] = LR_values_done_WP['R05_worry']
LR_values_done_WP['flood_experience'] = LR_values_done_WP['Q18_flood_exp']
LR_values_done_WP['perc_damage'] = LR_values_done_WP['R03_perc_damage']
LR_values_done_WP['self_efficacy'] = LR_values_done_WP['self_efficacy_WP']
LR_values_done_WP['resp_efficacy'] = LR_values_done_WP['resp_efficacy_WP']
LR_values_done_WP['perc_cost'] = LR_values_done_WP['perc_cost_WP']
LR_values_done_WP['done_other'] = LR_values_done_WP['done_DP']
remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP')
for k in remove:
    LR_values_done_WP.pop(k, None)
LR_values_done_WP

{'Intercept': -2.382891002312257,
 'Perc_probability': -0.3449500676077063,
 'worry': 0.8351560365134342,
 'flood_experience': 0.7248682368761468,
 'perc_damage': 0.15865663843752514,
 'self_efficacy': 0.6432834947530413,
 'resp_efficacy': 0.42686885537001773,
 'perc_cost': -1.4258860648665674,
 'done_other': 1.634870619083835}

In [21]:
# Wet-proofing
# logistic regression to 'plan_soon_WP'
# Define features (X) and target (y)
X_WP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP']]
y_WP_plan = data_DP_WP_reg['plan_soon_WP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_WP_plan, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_WP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 0.93

Logistic Regression Equation:
logit(p) = -2.5338 (0.3566 * R02_perc_prob) + (0.7271 * R05_worry) + (0.1332 * Q18_flood_exp) + (0.0342 * R03_perc_damage) + (2.0608 * self_efficacy_WP) + (0.3347 * resp_efficacy_WP) + (-1.9260 * perc_cost_WP) + (0.7853 * done_DP)

Feature Weights: {'R02_perc_prob': 0.3566241816188038, 'R05_worry': 0.7271413487031906, 'Q18_flood_exp': 0.13321058259589433, 'R03_perc_damage': 0.03417111011665783, 'self_efficacy_WP': 2.0607797434968456, 'resp_efficacy_WP': 0.3347035864798578, 'perc_cost_WP': -1.926012197448968, 'done_DP': 0.7853153772795295}
Intercept: -2.5338070940863404


In [22]:
LR_values_plan_soon_WP = dict(zip(feature_names, weights))
LR_values_plan_soon_WP['Intercept'] = intercept
LR_values_plan_soon_WP['Perc_probability'] = LR_values_plan_soon_WP['R02_perc_prob']
LR_values_plan_soon_WP['worry'] = LR_values_plan_soon_WP['R05_worry']
LR_values_plan_soon_WP['flood_experience'] = LR_values_plan_soon_WP['Q18_flood_exp']
LR_values_plan_soon_WP['perc_damage'] = LR_values_plan_soon_WP['R03_perc_damage']
LR_values_plan_soon_WP['self_efficacy'] = LR_values_plan_soon_WP['self_efficacy_WP']
LR_values_plan_soon_WP['resp_efficacy'] = LR_values_plan_soon_WP['resp_efficacy_WP']
LR_values_plan_soon_WP['perc_cost'] = LR_values_plan_soon_WP['perc_cost_WP']
LR_values_plan_soon_WP['done_other'] = LR_values_plan_soon_WP['done_DP']
remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_WP', 'resp_efficacy_WP', 'perc_cost_WP', 'done_DP')
for k in remove:
    LR_values_plan_soon_WP.pop(k, None)
LR_values_plan_soon_WP

{'Intercept': -2.5338070940863404,
 'Perc_probability': 0.3566241816188038,
 'worry': 0.7271413487031906,
 'flood_experience': 0.13321058259589433,
 'perc_damage': 0.03417111011665783,
 'self_efficacy': 2.0607797434968456,
 'resp_efficacy': 0.3347035864798578,
 'perc_cost': -1.926012197448968,
 'done_other': 0.7853153772795295}

In [23]:
# Dry-proofing
# logistic regression to 'done_DP'
# Define features (X) and target (y)
X_DP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP']]
y_DP_done = data_DP_WP_reg['done_DP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_DP_done, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_DP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 1.00

Logistic Regression Equation:
logit(p) = -4.6718 (-0.0588 * R02_perc_prob) + (0.3524 * R05_worry) + (0.2407 * Q18_flood_exp) + (0.3452 * R03_perc_damage) + (0.7431 * self_efficacy_DP) + (0.2379 * resp_efficacy_DP) + (-0.6505 * perc_cost_DP) + (6.1051 * done_WP)

Feature Weights: {'R02_perc_prob': -0.05880167341496487, 'R05_worry': 0.3524070245246781, 'Q18_flood_exp': 0.24070222891401938, 'R03_perc_damage': 0.34518478994107726, 'self_efficacy_DP': 0.7431351418058262, 'resp_efficacy_DP': 0.23785312212593318, 'perc_cost_DP': -0.6504909759726115, 'done_WP': 6.105131342052666}
Intercept: -4.67177283091222


In [24]:
LR_values_done_DP = dict(zip(feature_names, weights))
LR_values_done_DP['Intercept'] = intercept
LR_values_done_DP['Perc_probability'] = LR_values_done_DP['R02_perc_prob']
LR_values_done_DP['worry'] = LR_values_done_DP['R05_worry']
LR_values_done_DP['flood_experience'] = LR_values_done_DP['Q18_flood_exp']
LR_values_done_DP['perc_damage'] = LR_values_done_DP['R03_perc_damage']
LR_values_done_DP['self_efficacy'] = LR_values_done_DP['self_efficacy_DP']
LR_values_done_DP['resp_efficacy'] = LR_values_done_DP['resp_efficacy_DP']
LR_values_done_DP['perc_cost'] = LR_values_done_DP['perc_cost_DP']
LR_values_done_DP['done_other'] = LR_values_done_DP['done_WP']
remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP')
for k in remove:
    LR_values_done_DP.pop(k, None)
LR_values_done_DP

{'Intercept': -4.67177283091222,
 'Perc_probability': -0.05880167341496487,
 'worry': 0.3524070245246781,
 'flood_experience': 0.24070222891401938,
 'perc_damage': 0.34518478994107726,
 'self_efficacy': 0.7431351418058262,
 'resp_efficacy': 0.23785312212593318,
 'perc_cost': -0.6504909759726115,
 'done_other': 6.105131342052666}

In [25]:
# Dry-proofing
# logistic regression to 'plan_soon_DP'
# Define features (X) and target (y)
X_DP = data_DP_WP_reg[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP']]
y_DP_plan = data_DP_WP_reg['plan_soon_DP']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_WP, y_DP_plan, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.2f}")

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

# Print the logistic regression equation
feature_names = X_DP.columns
equation = f"logit(p) = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLogistic Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

Logistic Regression Model Accuracy: 0.91

Logistic Regression Equation:
logit(p) = -2.3562 (0.4870 * R02_perc_prob) + (1.1819 * R05_worry) + (0.5019 * Q18_flood_exp) + (0.0915 * R03_perc_damage) + (1.9037 * self_efficacy_DP) + (0.2593 * resp_efficacy_DP) + (-1.8763 * perc_cost_DP) + (-1.5433 * done_WP)

Feature Weights: {'R02_perc_prob': 0.4869996287709566, 'R05_worry': 1.1818606025882876, 'Q18_flood_exp': 0.5018731574508593, 'R03_perc_damage': 0.0914779314499942, 'self_efficacy_DP': 1.903701563840133, 'resp_efficacy_DP': 0.25930577464220494, 'perc_cost_DP': -1.876281775159464, 'done_WP': -1.543337404567993}
Intercept: -2.3562128753554874


In [26]:
LR_values_plan_soon_DP = dict(zip(feature_names, weights))
LR_values_plan_soon_DP['Intercept'] = intercept
LR_values_plan_soon_DP['Perc_probability'] = LR_values_plan_soon_DP['R02_perc_prob']
LR_values_plan_soon_DP['worry'] = LR_values_plan_soon_DP['R05_worry']
LR_values_plan_soon_DP['flood_experience'] = LR_values_plan_soon_DP['Q18_flood_exp']
LR_values_plan_soon_DP['perc_damage'] = LR_values_plan_soon_DP['R03_perc_damage']
LR_values_plan_soon_DP['self_efficacy'] = LR_values_plan_soon_DP['self_efficacy_DP']
LR_values_plan_soon_DP['resp_efficacy'] = LR_values_plan_soon_DP['resp_efficacy_DP']
LR_values_plan_soon_DP['perc_cost'] = LR_values_plan_soon_DP['perc_cost_DP']
LR_values_plan_soon_DP['done_other'] = LR_values_plan_soon_DP['done_WP']

remove = ('R02_perc_prob', 'R05_worry', 'Q18_flood_exp', 'R03_perc_damage', 'self_efficacy_DP', 'resp_efficacy_DP', 'perc_cost_DP', 'done_WP')
for k in remove:
    LR_values_plan_soon_DP.pop(k, None)
LR_values_plan_soon_DP

{'Intercept': -2.3562128753554874,
 'Perc_probability': 0.4869996287709566,
 'worry': 1.1818606025882876,
 'flood_experience': 0.5018731574508593,
 'perc_damage': 0.0914779314499942,
 'self_efficacy': 1.903701563840133,
 'resp_efficacy': 0.25930577464220494,
 'perc_cost': -1.876281775159464,
 'done_other': -1.543337404567993}

In [27]:
# combining the dictionaries to a dataframe
LogisticReg_PMT = pd.DataFrame([LR_values_done_WP, LR_values_plan_soon_WP, LR_values_done_DP, LR_values_plan_soon_DP], index=['done_WP', 'plan_soon_WP', 'done_DP', 'plan_soon_DP'])

# Transpose the DataFrame to have keys as index and dictionary names as columns
LogisticReg_PMT = LogisticReg_PMT.T

# Define the desired order of the index
desired_order = ['Intercept', 'worry', 'perc_damage', 'Perc_probability', 'flood_experience', 
                 'self_efficacy', 'resp_efficacy', 
                 'perc_cost', 'done_other'
                ]

# Reindex the DataFrame
LogisticReg_PMT = LogisticReg_PMT.reindex(desired_order)

LogisticReg_PMT

,done_WP,plan_soon_WP,done_DP,plan_soon_DP
Intercept,-2.382891,-2.533807,-4.671773,-2.356213
worry,0.835156,0.727141,0.352407,1.181861
perc_damage,0.158657,0.034171,0.345185,0.091478
Perc_probability,-0.344950,0.356624,-0.058802,0.487000
flood_experience,0.724868,0.133211,0.240702,0.501873
self_efficacy,0.643283,2.060780,0.743135,1.903702
resp_efficacy,0.426869,0.334704,0.237853,0.259306
perc_cost,-1.425886,-1.926012,-0.650491,-1.876282
done_other,1.634871,0.785315,6.105131,-1.543337


In [28]:
LogisticReg_PMT_done = LogisticReg_PMT[['done_WP', 'done_DP']]
LogisticReg_PMT_done = LogisticReg_PMT_done.rename(columns={"done_WP": "wet-proofing", "done_DP": "dry-proofing"})
LogisticReg_PMT_plan_soon = LogisticReg_PMT[['plan_soon_WP', 'plan_soon_DP']]
LogisticReg_PMT_plan_soon = LogisticReg_PMT_plan_soon.rename(columns={"plan_soon_WP": "wet-proofing", "plan_soon_DP": "dry-proofing"})
LogisticReg_PMT_done

,wet-proofing,dry-proofing
Intercept,-2.382891,-4.671773
worry,0.835156,0.352407
perc_damage,0.158657,0.345185
Perc_probability,-0.344950,-0.058802
flood_experience,0.724868,0.240702
self_efficacy,0.643283,0.743135
resp_efficacy,0.426869,0.237853
perc_cost,-1.425886,-0.650491
done_other,1.634871,6.105131


In [29]:
LogisticReg_PMT_done = LogisticReg_PMT_done.round(5)
LogisticReg_PMT_plan_soon = LogisticReg_PMT_plan_soon.round(5)

In [30]:
LogisticReg_PMT_done.to_csv('../../../data/processed/logistic_regression_PMT/Logistic_regression_PMT_done_NLUKw5.csv')
LogisticReg_PMT_plan_soon.to_csv('../../../data/processed/logistic_regression_PMT/Logistic_regression_PMT_plan_soon_NLUKw5.csv')